# Transformer basics with PyTorch

This notebook builds the central pieces of an encoder Transformer from small tensors. Run each cell and inspect every shape. No dataset download is required.

In [ ]:
import math
import torch
from torch import nn

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device:', device)

## 1. Token IDs and embeddings

A token ID is only an index. `nn.Embedding` looks up a learned vector for each index. Here `0` is padding.

In [ ]:
vocabulary = {'<pad>': 0, 'cash': 1, 'withdrawal': 2, 'missing': 3, 'card': 4}
input_ids = torch.tensor([[1, 2, 3], [4, 3, 0]], device=device)
attention_mask = (input_ids != vocabulary['<pad>']).long()
embedding = nn.Embedding(len(vocabulary), embedding_dim=8, padding_idx=0).to(device)
token_vectors = embedding(input_ids)
print('IDs:', input_ids.shape)
print('Embeddings:', token_vectors.shape)
print('Mask:\n', attention_mask)
print('Embedding parameters:', embedding.weight.numel())

## 2. Add position embeddings

Attention does not otherwise know whether a token is first or last. Learned position vectors have the same feature dimension and are added to token vectors.

In [ ]:
positions = torch.arange(input_ids.shape[1], device=device).unsqueeze(0)
position_embedding = nn.Embedding(16, 8).to(device)
x = token_vectors + position_embedding(positions)
print('Positions:', positions)
print('Combined representations:', x.shape)

## 3. Calculate one self-attention head

Learned projections create queries, keys, and values. `Q @ K.T` compares every token with every token; softmax turns the scores into weights.

In [ ]:
head_dimension = 4
query_layer = nn.Linear(8, head_dimension, bias=False).to(device)
key_layer = nn.Linear(8, head_dimension, bias=False).to(device)
value_layer = nn.Linear(8, head_dimension, bias=False).to(device)
Q, K, V = query_layer(x), key_layer(x), value_layer(x)
scores = Q @ K.transpose(-2, -1) / math.sqrt(head_dimension)
key_is_padding = ~attention_mask.bool().unsqueeze(1)
scores = scores.masked_fill(key_is_padding, float('-inf'))
weights = scores.softmax(dim=-1)
context = weights @ V
print('Q, K, V:', Q.shape, K.shape, V.shape)
print('Attention score matrix:', scores.shape)
print('First message weights:\n', weights[0].detach().cpu().round(decimals=3))
print('Context:', context.shape)

Each row of the weight matrix belongs to one query token and sums to 1. Padding columns receive zero probability after masking.

In [ ]:
print('Row sums:', weights.sum(dim=-1))
print('Attention paid to padded key:', weights[1, :, 2])

## 4. Use PyTorch multi-head attention

With model dimension 8 and two heads, each head handles four features. Their results are concatenated and projected back to dimension 8.

In [ ]:
multihead = nn.MultiheadAttention(8, num_heads=2, batch_first=True).to(device)
multihead_output, multihead_weights = multihead(
    x, x, x, key_padding_mask=~attention_mask.bool(), average_attn_weights=False
)
print('Output:', multihead_output.shape)
print('Weights [batch, heads, query, key]:', multihead_weights.shape)

## 5. Build a complete encoder layer

`TransformerEncoderLayer` combines multi-head attention, residual connections, LayerNorm, and a position-wise feed-forward network. Stacking layers lets representations become progressively more contextual.

In [ ]:
layer = nn.TransformerEncoderLayer(
    d_model=8, nhead=2, dim_feedforward=16, dropout=0.0,
    activation='gelu', batch_first=True, norm_first=True,
).to(device)
encoder = nn.TransformerEncoder(layer, num_layers=2, enable_nested_tensor=False).to(device)
encoded = encoder(x, src_key_padding_mask=~attention_mask.bool())
print('Encoder output:', encoded.shape)
print('Encoder parameters:', sum(p.numel() for p in encoder.parameters()))

## 6. Pool token features and classify

SupportRouter averages only real token positions, then maps the message representation to one logit per intent.

In [ ]:
valid = attention_mask.unsqueeze(-1).to(encoded.dtype)
pooled = (encoded * valid).sum(dim=1) / valid.sum(dim=1).clamp_min(1)
classifier = nn.Linear(8, 3).to(device)
logits = classifier(pooled)
probabilities = logits.softmax(dim=1)
print('Pooled messages:', pooled.shape)
print('Logits:', logits.shape)
print('Probabilities:\n', probabilities.detach().cpu())
print('Each row sums to:', probabilities.sum(dim=1).detach().cpu())

## 7. One optimization step

Cross-entropy receives raw logits. Backpropagation creates gradients for the classifier and encoder; AdamW uses them to update the weights.

In [ ]:
labels = torch.tensor([2, 1], device=device)
loss = nn.CrossEntropyLoss()(logits, labels)
loss.backward()
print('Loss:', loss.item())
print('Classifier gradient shape:', classifier.weight.grad.shape)
print('Gradient norm:', classifier.weight.grad.norm().item())

## Exercises

1. Change the second message so it has no padding. How do the mask and attention weights change?
2. Remove the `sqrt(head_dimension)` scaling and compare the attention weights.
3. Change from two to four heads while keeping dimension 8. What is each head's dimension?
4. Increase the feed-forward dimension and calculate how many parameters are added.
5. Read `src/support_router/models.py` and map each operation to a section of this notebook.